## Imports

In [1]:
import os

from safetensors.torch import load_file
import re
from datasets import load_dataset, Dataset
from transformers import BertTokenizer, BertModel, Trainer, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint
import torch
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, matthews_corrcoef
import wandb

C:\Users\anton\PycharmProjects\AML-miniproject-ERNIE\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Consts

In [2]:
# Hyperparameters
BATCH_SIZE = 32
LEARNING_RATE = 2e-5
MAX_SEQ_LENGTH = 512
epochs = 3

# Configurations for experiment tracking
record_stats = False  # Set to True to enable Weights & Biases logging
test_model = False    # Set to True to skip training and only run evaluation and prediction (make sure to have a checkpoint in ./results)
model_name = "google-bert/bert-base-uncased"
data_set_name = "stanfordnlp/imdb"
wandb_run_name = f"bert-base-uncased-imdb-bs{BATCH_SIZE}-lr{LEARNING_RATE}-ep{epochs}"
wandb_project_name = "aml-miniproject-ernie"

## Load and split data (and Tokenize)

In [3]:
# Data Retrieval
data = load_dataset("stanfordnlp/imdb")
# Filter dataset html tags using regex
data = data.map(lambda x: {"text": re.sub(r"<.*?>", "", x["text"])})  # Remove HTML tags from the text

# Remove duplicates from the training set
df = data["train"].to_pandas().drop_duplicates(subset="text")
data["train"] = Dataset.from_pandas(df)

train_data, test_data = data["train"], data["test"]

# Shuffle the training data to ensure validation set isn't biased of order
train_data = train_data.shuffle(seed=42)

# Create a validation set from the training data
split_idx = int(0.9 * len(train_data))
val_data = train_data.select(range(split_idx, len(train_data)))
train_data = train_data.select(range(split_idx))

# Initialize the tokenizer
tokenizer = BertTokenizer.from_pretrained(model_name)

def tokenize(batch):
    # Normal Hugging Face tokenization: truncates from the end and pads to MAX_SEQ_LENGTH
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_SEQ_LENGTH,
    )

# Tokenize the datasets (this gives us input_ids, attention_mask, and token_type_ids)
train_data = train_data.map(tokenize, batched=True)
test_data = test_data.map(tokenize, batched=True)
val_data = val_data.map(tokenize, batched=True)

# Set the format for PyTorch
train_data.set_format(type="torch", columns=["input_ids", "attention_mask", "token_type_ids", "label"])
test_data.set_format(type="torch", columns=["input_ids", "attention_mask", "token_type_ids", "label"])
val_data.set_format(type="torch", columns=["input_ids", "attention_mask", "token_type_ids", "label"])

## Showcase tokenizer (example)

In [ ]:
# Showcase tokenization on a sample review
sample = "This movie was amazing!"
tokens = tokenizer.tokenize(sample)
enc = tokenizer(sample, return_tensors="pt")
token_ids = enc["input_ids"]
print("Tokens:", tokens)
# ['this', 'movie', 'was', 'amazing', '!']
print("With special tokens:", tokenizer.convert_ids_to_tokens(enc["input_ids"][0]))
# ['[CLS]', 'this', 'movie', 'was', 'amazing', '!', '[SEP]']
print("token_ids:", token_ids)
print("token_type_ids:", enc["token_type_ids"])  # all zeros for single sequence

## Look for languages in dataset

In [4]:
from langdetect import detect
import re
def detect_language(text):
    try:
        text['lang'] = detect(text['text'])
    except:
        text['lang'] = 'unknown'
    return text
language_results = data["train"].map(detect_language)
from collections import Counter
language_counts = Counter(language_results["lang"])
print("languages in data: ", language_counts)

# get lengths of reviews (maybe implement head+tail tokenization if we find very long reviews)
lengths = [len(tokenizer.tokenize(text)) for text in data["train"]["text"]]
print("Average review length (in tokens):", np.mean(lengths))
print("Max review length (in tokens):", np.max(lengths))
print("Min review length (in tokens):", np.min(lengths))
print("Reviews longer than 512 tokens:", sum(l > 512 for l in lengths))




Average review length (in tokens): 295.87632012207365
Max review length (in tokens): 3053
Min review length (in tokens): 11
Reviews longer than 512 tokens: 3286


In [5]:
# look for formatting issues in reviews (e.g., presence of HTML tags)
def check_formatting(text):
    if re.search(r'<.*?>', text):
        print("Review with potential formatting issue:", text)
        return True
    return False
formatting_issues = sum(check_formatting(text) for text in data["train"]["text"])
print("Reviews with potential formatting issues:", formatting_issues)

# Check for duplicates in training data
df = train_data.to_pandas()
print("Number of duplicates in dataset: ", df["text"].duplicated().sum())  # Check for exact duplicates

Reviews with potential formatting issues: 0
Number of duplicates in dataset:  0


## Define and load BERT model for Sequence classification

In [6]:
# Model Definition
class BertForSentiment(torch.nn.Module):
    def __init__(self, model_name, num_labels=2):
        super().__init__()
        self.bert = BertModel.from_pretrained(model_name)
        self.dropout = torch.nn.Dropout(0.1)
        # ← OUTPUT HEAD: change num_labels to adapt to a different task
        self.classifier = torch.nn.Linear(self.bert.config.hidden_size, num_labels)
        self.num_labels = num_labels

    def forward(self, input_ids, attention_mask, token_type_ids=None, labels=None):
        # ← INPUT HEAD: pass token_type_ids here for two-sequence tasks
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

        # ← Take CLS token for sequence classification
        cls_output = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(self.dropout(cls_output))

        # Compute loss if labels are provided (needed for Trainer compatibility)
        loss = None
        if labels is not None:
            loss = torch.nn.CrossEntropyLoss()(logits, labels)

        # Return in the format the Trainer expects
        from transformers.modeling_outputs import SequenceClassifierOutput
        return SequenceClassifierOutput(loss=loss, logits=logits)

model = BertForSentiment(model_name)


BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Show embedding layers of the model

In [ ]:
# Show the embedding layers explicitly
embeddings = model.bert.embeddings
print("Word embedding shape:", embeddings.word_embeddings.weight.shape)
print("Position embedding shape:", embeddings.position_embeddings.weight.shape)

# Show what a CLS embedding actually looks like
with torch.no_grad():
    enc = tokenizer("Brilliant film!", return_tensors="pt")
    raw_embeddings = model.bert.embeddings(enc["input_ids"])
    print("CLS embedding (first 8 dims):", raw_embeddings[0, 0, :8])

# Freeze model weights (Optional)

In [ ]:
# Freeze model weights (except the classifier head) to speed up Backpropagation
for param in model.bert.parameters():
    param.requires_grad = False

### Load old model if you have one you want to reuse (don't run if you don't have a model)

In [ ]:
last_checkpoint = get_last_checkpoint("./results")
print(f"Loading model from checkpoint: {last_checkpoint}")
state_dict = load_file(os.path.join(last_checkpoint, "model.safetensors"))
model = BertForSentiment(model_name)
model.load_state_dict(state_dict)

## Select runtime for model

In [7]:
# Select one runtime device and keep model/inputs aligned.
if torch.cuda.is_available():
    device = torch.device("cuda") # NVIDIA GPU support
elif torch.backends.mps.is_available():
    device = torch.device("mps") # Apple Silicon GPU support
else:
    device = torch.device("cpu")
model.to(device)

BertForSentiment(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_

# Define metrics

In [8]:
# Define a compute_metrics function for the Trainer to use during evaluation
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)
    mcc = matthews_corrcoef(labels, preds)
    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "mcc": mcc,
    }

# Setup training arguments

In [9]:
# Set up training arguments for the Hugging Face Trainer
training_args = TrainingArguments(
    num_train_epochs=epochs,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    eval_strategy="epoch",                               # Evaluate the model at the end of each epoch
    save_strategy="epoch",                               # Save the model at the end of each epoch
    save_total_limit=2,                                  # Only keep the 2 most recent checkpoints to save disk space
    weight_decay=0.01,                                   # Regularization to prevent overfitting by adding a penalty to the loss function based on the magnitude of the model's weights.
    learning_rate=LEARNING_RATE,
    logging_steps=50,
    logging_strategy="steps",                            # Log training metrics every 50 steps
    report_to=["wandb"] if record_stats else "none",     # Log to Weights & Biases if enabled
    output_dir="./results",                              # Directory to save model checkpoints and logs
    run_name=wandb_run_name if record_stats else None,
    fp16=device.type == "cuda",                          # Use mixed precision only on CUDA (if gpu is available)
    load_best_model_at_end=True,                         # Load the best model at the end of training based on evaluation metrics
    metric_for_best_model="f1",                          # Use F1 score to determine the best model
)

# Sets optimizer to AdamW by default
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    compute_metrics=compute_metrics,
)

# Run training

In [10]:
trainer.train()
wandb.finish()

TrainOutput(global_step=2103, training_loss=0.14301089764250724, metrics={'train_runtime': 512.7291, 'train_samples_per_second': 131.134, 'train_steps_per_second': 4.102, 'total_flos': 0.0, 'train_loss': 0.14301089764250724, 'epoch': 3.0})

# Run evaluation

In [11]:
# Fix for Transformers notebooks: the NotebookProgressCallback can crash on evaluate() if you haven't trained in this session.
# (Error: "on_train_begin must be called before on_evaluate")
try:
    from transformers.utils.notebook import NotebookProgressCallback
    trainer.remove_callback(NotebookProgressCallback)
except Exception:
    pass

evaluation = trainer.evaluate(eval_dataset=test_data)
print("Evaluation results:", evaluation)

KeyboardInterrupt: 

# Manually test model

In [ ]:
def predict_review(text):
    model.eval()
    model_device = next(model.parameters()).device

    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=MAX_SEQ_LENGTH,
    )
    enc = {k: v.to(model_device) for k, v in enc.items()}

    with torch.no_grad():
        out = model(**enc)
        probs = torch.softmax(out.logits, dim=1).squeeze(0)
        pred_id = torch.argmax(probs).item()

    return {
        "label": pred_id,
        "confidence": float(probs[pred_id].item()),
        "prob_negative": float(probs[0].item()),
        "prob_positive": float(probs[1].item()),
    }

while True:
    txt = input("Enter a movie review (or 'quit'): ").strip()
    if txt.lower() == "quit":
        break
    result = predict_review(txt)
    print(result)



